# Convert raw FR5 episodes → LeRobot, then push to the Hub

Your recordings live on the Hub as **raw capture folders** (`episode_XXXX/` with
`data.csv` + `wrist_cam.mp4` + `scene_cam.mp4` + `meta.json`) — e.g.
`Slifold/episodes_20260717` (91 episodes). The **training** notebooks
(`train_pi0/pi05/pi0_fast_runpod.ipynb`) consume the **LeRobot v3.0** format
(`meta/info.json`, `data/chunk-000/*.parquet`, `frames/…`), so the raw data must be
converted **once** and pushed as a separate dataset repo the training notebooks point at.

This notebook: pull raw → run the repo's tested `common/convert_episodes.py`
(fetched, pinned to commit `a518b59c0fc5`) → verify → push the LeRobot dataset.

It produces: **7-D joint+gripper state**, **7-D action** (6 cmd joints + gripper), the
**wrist camera** only, **30 fps** (resampled from the ~40–55 Hz capture), and a default
task string where `meta.json`'s instruction is empty.

Run it on any machine with internet + ~6 GB free disk (a small CPU pod is plenty — no GPU needed).

## 1 · Parameters

In [ ]:
import os

HF_TOKEN          = os.environ.get("HF_TOKEN", "")               # hf_... (read source, write target)
SOURCE_RAW_REPO   = "Slifold/episodes_20260717"                  # raw episodes (input)
TARGET_LEROBOT_REPO = "<you>/fr5-pick-place-lerobot"             # converted dataset (output, created private)

CAMERAS       = "wrist,scene"   # both views (every policy uses 2 cams) | "wrist" for single-cam
TASK_OVERRIDE = ""          # "" -> converter default ("pick up the block and place it in the bin")
EXTRACT_STRIDE = 1          # keep every Kth start frame (1 = all; raise for lighter datasets)

RAW_DIR  = "/tmp/raw_episodes"
OUT_DIR  = "/tmp/lerobot_dataset"

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF token (hf_...): ").strip()
assert HF_TOKEN.startswith("hf_"), "need an HF token (read source + write target)"
assert TARGET_LEROBOT_REPO != "<you>/fr5-pick-place-lerobot", "set TARGET_LEROBOT_REPO"
print("parameters set")

## 2 · Install dependencies

Just what the converter needs — no torch/lerobot. Non-quiet so conflicts are visible.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install",
                "huggingface_hub", "numpy", "pandas", "pyarrow",
                "opencv-python-headless"], check=True)
print("deps ready")

## 3 · HuggingFace login

In [ ]:
from huggingface_hub import login, whoami
login(token=HF_TOKEN, add_to_git_credential=False)
print("logged in as:", whoami()["name"])

## 4 · Download the raw episodes

In [ ]:
from huggingface_hub import snapshot_download
import pathlib

snapshot_download(SOURCE_RAW_REPO, repo_type="dataset", local_dir=RAW_DIR)
eps = sorted(p.name for p in pathlib.Path(RAW_DIR).glob("episode_*") if p.is_dir())
print(f"{len(eps)} episodes downloaded, e.g. {eps[:3]} ... {eps[-1] if eps else ''}")
assert eps, f"no episode_* folders under {RAW_DIR}"

## 5 · Fetch the tested converter

Pulls `common/convert_episodes.py` from the public repo, **pinned to commit
`a518b59c0fc5`** for reproducibility (not `main`, which moves). It is self-contained
(numpy/pandas/pyarrow/cv2 only).

In [ ]:
import urllib.request

RAW = "https://raw.githubusercontent.com/SreevaatsavB/fairino-fr5-policies/a518b59c0fc5337378c8b500b6502cfef35d15b9/common/convert_episodes.py"
urllib.request.urlretrieve(RAW, "convert_episodes.py")
print("fetched convert_episodes.py (pinned @ a518b59c0fc5)")

## 6 · Convert raw → LeRobot

`--extract-frames` writes per-frame JPEGs (what the training notebooks' inline
`FR5Dataset` reads). Streams progress; a 91-episode set takes a few minutes.

In [ ]:
import subprocess, sys

cmd = [sys.executable, "convert_episodes.py",
       "--episodes", RAW_DIR, "--out", OUT_DIR,
       "--extract-frames", "--cameras", CAMERAS,
       "--extract-stride", str(EXTRACT_STRIDE)]
if TASK_OVERRIDE:
    cmd += ["--task", TASK_OVERRIDE]
subprocess.run(cmd, check=True)

## 7 · Verify the converted dataset

In [ ]:
import json, pathlib
info = json.loads(pathlib.Path(OUT_DIR, "meta", "info.json").read_text())
sd = info["features"]["observation.state"]["shape"][0]
ad = info["features"]["action"]["shape"][0]
cams = [k for k in info["features"] if k.startswith("observation.images.")]
print(f"episodes = {info['total_episodes']}   frames = {info['total_frames']}   fps = {info['fps']}")
print(f"state_dim = {sd}   action_dim = {ad}   cameras = {cams}")
assert sd == 7 and ad == 7, f"expected 7/7, got {sd}/{ad}"
assert "observation.images.wrist_cam" in cams
tasks = pathlib.Path(OUT_DIR, "meta", "tasks.parquet")
if tasks.exists():
    import pyarrow.parquet as pq
    print("tasks:", pq.read_table(tasks).to_pandas()["task"].tolist())

### 7b · Eyeball one episode

In [ ]:
import pandas as pd, pathlib
import matplotlib
try:
    import matplotlib.pyplot as plt, matplotlib.image as mpimg
    df = pd.read_parquet(pathlib.Path(OUT_DIR, "data/chunk-000/file-000.parquet"))
    ep = df[df.episode_index == 0]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    frames = sorted(pathlib.Path(OUT_DIR, "frames").rglob("ep-000/*.jpg"))
    if frames:
        ax[0].imshow(mpimg.imread(frames[len(frames)//2])); ax[0].axis("off")
        ax[0].set_title(f"wrist cam ({len(frames)} frames)")
    pd.DataFrame(ep["action"].tolist()).plot(ax=ax[1], legend=False,
        title=f"episode 0 action traces (7-D, {len(ep)} steps)")
    plt.tight_layout(); plt.show()
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib"], check=True)
    print("installed matplotlib — re-run this cell to see the plot")

## 8 · Push the LeRobot dataset to the Hub

Creates `TARGET_LEROBOT_REPO` private and uploads. Re-running syncs changes.

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(TARGET_LEROBOT_REPO, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=OUT_DIR, repo_id=TARGET_LEROBOT_REPO, repo_type="dataset",
                  commit_message="converted FR5 episodes -> LeRobot v3.0")
print(f"pushed -> https://huggingface.co/datasets/{TARGET_LEROBOT_REPO}")
print(f"\nNow in the training notebooks set:  HF_DATASET_REPO = \"{TARGET_LEROBOT_REPO}\"")